In [25]:
import os
import psycopg2
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

url = "https://api.coinpaprika.com/v1/tickers?quotes=USD"

response = requests.get(url)

print(response)

<Response [200]>


In [26]:
data = response.json()

data[0]

{'id': 'btc-bitcoin',
 'name': 'Bitcoin',
 'symbol': 'BTC',
 'rank': 1,
 'total_supply': 20053962,
 'max_supply': 21000000,
 'beta_value': 0.973337,
 'first_data_at': '2010-07-17T00:00:00Z',
 'last_updated': '2026-07-09T06:14:16Z',
 'quotes': {'USD': {'price': 62473.18282251164,
   'volume_24h': 23018573616.45167,
   'volume_24h_change_24h': -18.139999389648438,
   'market_cap': 1252834084663,
   'market_cap_change_24h': -0.25,
   'percent_change_15m': -0.25999999046325684,
   'percent_change_30m': 0.12999999523162842,
   'percent_change_1h': 0.3199999928474426,
   'percent_change_6h': 0.5899999737739563,
   'percent_change_12h': 0.49000000953674316,
   'percent_change_24h': -0.25,
   'percent_change_7d': 3.859999895095825,
   'percent_change_30d': 0,
   'percent_change_1y': 0,
   'ath_price': 126173.1777846797,
   'ath_date': '2025-10-06T19:00:40Z',
   'percent_from_price_ath': -50.49}}}

In [27]:


# coin_df = pd.DataFrame(data)

# coin_df.head(5)

rows = []
for coin in data:
    if coin["symbol"] in ["BTC","ETH","USDT","BNB","USDC","GAL","UTK","MAPO","BTX"]:
        rows.append({
            "name":coin["name"],
            "symbol": coin["symbol"],
            "rank": coin["rank"],
            "last_update": coin["last_updated"],
            "price_usd" : coin["quotes"]["USD"]["price"],
            "mrkt_cap" : coin["quotes"]["USD"]["market_cap"]

        })

coins_df = pd.DataFrame(rows)

coins_df

,name,symbol,rank,last_update,price_usd,mrkt_cap
0,Bitcoin,BTC,1,2026-07-09T06:14:16Z,62473.182823,1252834084663
1,Ethereum,ETH,2,2026-07-09T06:14:16Z,1746.111425,210277766238
2,Tether,USDT,3,2026-07-09T06:14:16Z,0.999297,184165431567
3,BNB,BNB,4,2026-07-09T06:14:16Z,572.534226,77167695681
4,USDC,USDC,5,2026-07-09T06:14:16Z,1.000032,73213965529
5,CCIP Bridged USDC (Ronin),USDC,621,2026-07-09T06:14:16Z,0.999128,19194827
6,xMoney,UTK,979,2026-07-09T06:14:16Z,0.008165,5748772
7,Galatasaray Fan Token,GAL,983,2026-07-09T06:14:16Z,0.874915,5746918
8,MAP Protocol,MAPO,984,2026-07-09T06:14:16Z,0.001103,5687353
9,BeatSwap,BTX,1027,2026-07-09T06:14:16Z,0.022406,5035953


In [28]:
coins_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         14 non-null     str    
 1   symbol       14 non-null     str    
 2   rank         14 non-null     int64  
 3   last_update  14 non-null     str    
 4   price_usd    14 non-null     float64
 5   mrkt_cap     14 non-null     int64  
dtypes: float64(1), int64(2), str(3)
memory usage: 804.0 bytes


In [29]:
coins_df["last_update"] = pd.to_datetime(coins_df["last_update"])
coins_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   name         14 non-null     str                
 1   symbol       14 non-null     str                
 2   rank         14 non-null     int64              
 3   last_update  14 non-null     datetime64[us, UTC]
 4   price_usd    14 non-null     float64            
 5   mrkt_cap     14 non-null     int64              
dtypes: datetime64[us, UTC](1), float64(1), int64(2), str(2)
memory usage: 804.0 bytes


In [30]:
connection_to_db = psycopg2.connect(
    host= os.getenv("host"),
    port = os.getenv("port"),
    database = os.getenv("database"),
    user = os.getenv("user"),
    password = os.getenv("password")
)